# 07 - K-Means Clustering
### Credit Portfolio Segmentation and Risk Pattern Analysis

---

## 📌 Beginner Objective
**K-Means** is the most popular clustering algorithm in industry:
1. It partitions borrowers into $k$ groups.
2. It assigns each borrower to the nearest cluster center (centroid).
3. We use the **Elbow Method** to choose the best number of clusters ($k$).
4. We assign each borrower their cluster label and visualize them in our 2D PCA space.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Load scaled features and full dataset
X_scaled = pd.read_csv("../data/processed/scaled_features.csv")
df = pd.read_csv("../data/processed/cleaned_loans_dataset.csv")

print(f"Ready to cluster {len(X_scaled):,} borrowers.")


---
## 1. The Elbow Method (Finding the Best $k$)
We test $k$ from 2 to 7 and record the **Inertia** (sum of squared distances to cluster centers).
When adding more clusters stops significantly reducing inertia, we have found the "elbow".


In [ ]:
inertias = []
k_range = range(2, 8)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertias, marker='s', color='teal', linewidth=2)
plt.title("The Elbow Method for Optimal k")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia (Within-Cluster Sum of Squares)")
plt.tight_layout()
plt.savefig("../outputs/figures/kmeans_elbow_curve.png", dpi=300)
plt.show()


---
## 2. Fit K-Means with Optimal $k=4$
The elbow curve flattens significantly around $k=4$. Let's fit our final K-Means model!


In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['KMeans_Cluster'] = kmeans.fit_predict(X_scaled)

print("Borrower distribution across clusters:")
print(df['KMeans_Cluster'].value_counts().sort_index())


---
## 3. Visualize K-Means Clusters in 2D PCA Space


In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df['PC1'] = X_pca[:, 0]
df['PC2'] = X_pca[:, 1]

plt.figure(figsize=(10, 6))
sns.scatterplot(
    x='PC1', y='PC2', 
    hue='KMeans_Cluster', 
    data=df.sample(2500, random_state=42), 
    palette='Set1', 
    alpha=0.7
)
plt.title("Borrower Clusters Identified by K-Means (in PCA Space)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Cluster")

plt.tight_layout()
plt.savefig("../outputs/figures/kmeans_clusters_pca.png", dpi=300)
plt.show()
